In [3]:
%cd NLP-Based-Singlish-Need-Classification-System-

/content/NLP-Based-Singlish-Need-Classification-System-


In [4]:
!ls

 dashboard   data  'nlp service'


In [5]:
%cd "nlp service"
!ls

/content/NLP-Based-Singlish-Need-Classification-System-/nlp service
augment_data.py		  NLP_Based_Disaster_Need_Classification.ipynb
baseline_tokenizer.model  __pycache__
baseline_tokenizer.vocab  README.md
build_robustness_set.py   requirements.txt
clean_data.py		  split_data.py
hybrid_tokenizer.py	  train_tokenizer.py
make_train_txt.py


In [6]:
!pwd
!git status

/content/NLP-Based-Singlish-Need-Classification-System-/nlp service
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [7]:
!pip install transformers peft

In [1]:
%%writefile hybrid_tokenizer.py
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
MAX_LEN = 64

def hybrid_encode(text, oov_threshold=0.4):
    ids = [tokenizer.bos_token_id]
    for word in text.strip().split():
        pieces = tokenizer.tokenize(word)
        fragmentation = len(pieces) / max(len(word), 1)
        if fragmentation > oov_threshold or tokenizer.unk_token in pieces:
            for ch in word:
                ch_ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(ch))
                ids.extend(ch_ids if ch_ids else [tokenizer.unk_token_id])
        else:
            ids.extend(tokenizer.convert_tokens_to_ids(pieces))
    ids.append(tokenizer.eos_token_id)
    ids = ids[:MAX_LEN]
    attention_mask = [1] * len(ids)
    pad_len = MAX_LEN - len(ids)
    ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return ids, attention_mask

def baseline_encode(text):
    out = tokenizer(text, padding="max_length", truncation=True, max_length=MAX_LEN)
    return out["input_ids"], out["attention_mask"]

Writing hybrid_tokenizer.py


In [2]:
%%writefile model.py
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
import torch
import torch.nn as nn

NEED_KEYWORDS = ["one", "epa", "please", "ikmanin", "ikmanata", "help",
                  "urgent", "puluwanda", "asaneepa"]

def get_aux_features(text):
    words = text.lower().split()
    length_feature = min(len(words) / 20, 1.0)
    keyword_hits = sum(1 for w in words if w in NEED_KEYWORDS)
    keyword_feature = min(keyword_hits / 3, 1.0)
    return [length_feature, keyword_feature]

class SinglishNeedClassifier(nn.Module):
    def __init__(self, num_categories=5, aux_feature_dim=2):
        super().__init__()
        base = AutoModel.from_pretrained("xlm-roberta-base")
        lora_config = LoraConfig(
            r=8, lora_alpha=16,
            target_modules=["query", "value"],
            lora_dropout=0.1,
        )
        self.encoder = get_peft_model(base, lora_config)
        hidden = base.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden + aux_feature_dim, 128),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_categories),
        )

    def forward(self, input_ids, attention_mask, aux_features):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        combined = torch.cat([pooled, aux_features], dim=1)
        return self.classifier(combined)

Writing model.py


In [3]:
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from hybrid_tokenizer import hybrid_encode, baseline_encode
from model import SinglishNeedClassifier, get_aux_features

In [9]:
!ls /content



hybrid_tokenizer.py  model.py  __pycache__  sample_data


hybrid_tokenizer.py				__pycache__
model.py					sample_data
NLP-Based-Singlish-Need-Classification-System-


In [12]:
!mv hybrid_tokenizer.py model.py "NLP-Based-Singlish-Need-Classification-System-/nlp service/"

In [13]:
%cd "NLP-Based-Singlish-Need-Classification-System-/nlp service"
!pwd
!git status

/content/NLP-Based-Singlish-Need-Classification-System-/nlp service
/content/NLP-Based-Singlish-Need-Classification-System-/nlp service
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   hybrid_tokenizer.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	model.py

no changes added to commit (use "git add" and/or "git commit -a")


In [14]:
!git config --global user.email "malshagonaduwa723@gmail.com"
!git config --global user.name "malsha890"
!git add hybrid_tokenizer.py model.py
!git commit -m "Saved working tokenizer and model as proper modules✏️"
!git push

[main 7e6485f] Saved working tokenizer and model as proper modules✏️
 2 files changed, 63 insertions(+), 31 deletions(-)
 rewrite nlp service/hybrid_tokenizer.py (89%)
 create mode 100644 nlp service/model.py
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.48 KiB | 1.48 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/malsha890/NLP-Based-Singlish-Need-Classification-System-.git
   fa3108b..7e6485f  main -> main


In [15]:
from sklearn.model_selection import train_test_split

train_full = pd.read_csv("../data/train_augmented.csv")
train_final, val_final = train_test_split(
    train_full, test_size=0.15, stratify=train_full["category"], random_state=42
)

In [17]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [5]:
import torch
from torch.utils.data import Dataset

CATEGORIES = ["medical aid", "shelter", "food/water", "rescue/missing", "other"]
LABEL2ID = {c: i for i, c in enumerate(CATEGORIES)}

class NeedDataset(Dataset):
    def __init__(self, dataframe, encode_fn):
        self.df = dataframe.reset_index(drop=True)
        self.encode_fn = encode_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids, attention_mask = self.encode_fn(row["text"])
        aux = get_aux_features(row["text"])
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "aux_features": torch.tensor(aux, dtype=torch.float),
            "label": torch.tensor(LABEL2ID[row["category"]]),
        }

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_full = pd.read_csv("../data/train_augmented.csv")
train_final, val_final = train_test_split(
    train_full, test_size=0.15, stratify=train_full["category"], random_state=42
)

print(f"train_final: {len(train_final)}, val_final: {len(val_final)}")

train_final: 892, val_final: 158


In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_counts = train_final["category"].value_counts()
classes_ordered = [CATEGORIES[i] for i in range(len(CATEGORIES))]
weights = compute_class_weight("balanced", classes=np.array(classes_ordered), y=train_final["category"])
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to("cuda")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

In [11]:
from sklearn.metrics import f1_score
import numpy as np

test_clean = pd.read_csv("../data/test_clean.csv")
test_robust = pd.read_csv("../data/test_robustness.csv")

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

def evaluate(model, dataframe, encode_fn):
    dataset = NeedDataset(dataframe, encode_fn)
    loader = DataLoader(dataset, batch_size=16)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to("cuda"),
                            batch["attention_mask"].to("cuda"),
                            batch["aux_features"].to("cuda"))
            preds.extend(logits.argmax(dim=1).cpu().tolist())
            labels.extend(batch["label"].tolist())
    return f1_score(labels, preds, average="macro")

def train_one_run(encode_fn, seed, epochs=8):
    set_seed(seed)
    model = SinglishNeedClassifier().to("cuda")
    optimizer = AdamW(model.parameters(), lr=2e-4)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(NeedDataset(train_final, encode_fn), batch_size=16, shuffle=True)
    best_val_f1 = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch["input_ids"].to("cuda"),
                            batch["attention_mask"].to("cuda"),
                            batch["aux_features"].to("cuda"))
            loss = criterion(logits, batch["label"].to("cuda"))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        val_f1 = evaluate(model, val_final, encode_fn)
        print(f"  epoch {epoch}: train loss = {avg_loss:.4f}, val macro-F1 = {val_f1:.3f}")
        best_val_f1 = max(best_val_f1, val_f1)

    test_f1 = evaluate(model, test_clean, encode_fn)
    robust_f1 = evaluate(model, test_robust, encode_fn)
    return test_f1, robust_f1

SEEDS = [13, 42, 77, 101, 256]
results = []

for name, encode_fn in [("baseline", baseline_encode), ("hybrid", hybrid_encode)]:
    for seed in SEEDS:
        print(f"Running {name}, seed {seed}...")
        test_f1, robust_f1 = train_one_run(encode_fn, seed)
        results.append({"tokenizer": name, "seed": seed, "test_f1": test_f1, "robustness_f1": robust_f1})

results_df = pd.DataFrame(results)
results_df.to_csv("../data/results.csv", index=False)
print(results_df)

Running baseline, seed 13...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 0: train loss = 1.6273, val macro-F1 = 0.067
  epoch 1: train loss = 1.6056, val macro-F1 = 0.357
  epoch 2: train loss = 1.4999, val macro-F1 = 0.612
  epoch 3: train loss = 1.1117, val macro-F1 = 0.666
  epoch 4: train loss = 0.8116, val macro-F1 = 0.782
  epoch 5: train loss = 0.6535, val macro-F1 = 0.812
  epoch 6: train loss = 0.5189, val macro-F1 = 0.875
  epoch 7: train loss = 0.4227, val macro-F1 = 0.911
Running baseline, seed 42...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 0: train loss = 1.6201, val macro-F1 = 0.181
  epoch 1: train loss = 1.6069, val macro-F1 = 0.313
  epoch 2: train loss = 1.5778, val macro-F1 = 0.322
  epoch 3: train loss = 1.4747, val macro-F1 = 0.658
  epoch 4: train loss = 1.0882, val macro-F1 = 0.739
  epoch 5: train loss = 0.8054, val macro-F1 = 0.806
  epoch 6: train loss = 0.6097, val macro-F1 = 0.898
  epoch 7: train loss = 0.4694, val macro-F1 = 0.905
Running hybrid, seed 256...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 0: train loss = 1.6153, val macro-F1 = 0.077
  epoch 1: train loss = 1.5983, val macro-F1 = 0.332
  epoch 2: train loss = 1.5730, val macro-F1 = 0.440
  epoch 3: train loss = 1.3275, val macro-F1 = 0.630
  epoch 4: train loss = 1.0144, val macro-F1 = 0.735
  epoch 5: train loss = 0.7721, val macro-F1 = 0.817
  epoch 6: train loss = 0.6128, val macro-F1 = 0.905
  epoch 7: train loss = 0.4773, val macro-F1 = 0.899
  tokenizer  seed   test_f1  robustness_f1
0  baseline    13  0.832252       0.699389
1  baseline    42  0.781516       0.640724
2  baseline    77  0.841813       0.666196
3  baseline   101  0.782423       0.709252
4  baseline   256  0.853111       0.716231
5    hybrid    13  0.885556       0.728667
6    hybrid    42  0.816162       0.711842
7    hybrid    77  0.842298       0.596624
8    hybrid   101  0.740686       0.686843
9    hybrid   256  0.708263       0.730651


In [12]:
summary = results_df.groupby("tokenizer")[["test_f1", "robustness_f1"]].agg(["mean", "std"])
print(summary)

            test_f1           robustness_f1          
               mean       std          mean       std
tokenizer                                            
baseline   0.818223  0.033910      0.686359  0.031919
hybrid     0.798593  0.072962      0.690925  0.055564


In [13]:
from google.colab import files
files.download("../data/results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
results_df.to_csv("../data/results.csv", index=False)
!git add ../data/results.csv
!git commit -m "final results: baseline vs hybrid, 5 seeds, corrected LR and class weighting"
!git push

[main 5bbab72] final results: baseline vs hybrid, 5 seeds, corrected LR and class weighting
 1 file changed, 11 insertions(+)
 create mode 100644 data/results.csv
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 653 bytes | 653.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/malsha890/NLP-Based-Singlish-Need-Classification-System-.git
   7e6485f..5bbab72  main -> main


In [18]:
import os

In [19]:
token = os.getenv("GITHUB_TOKEN")